<a href="https://colab.research.google.com/github/Py-saqlain/Movie_Recap/blob/main/MovieApp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Connect to your Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Check the GPU
print("\n--- GPU CHECK ---")
!nvidia-smi -L

# 3. Check for your Movie files
import os
folder_path = '/content/drive/MyDrive/MovieApp'
print("\n--- FILE CHECK ---")
try:
    files = os.listdir(folder_path)
    print(f"SUCCESS! The supercomputer sees your folder! Files inside: {files}")
except FileNotFoundError:
    print(f"ERROR: Still can't find {folder_path}.")

In [ ]:
# Install zstd dependency for Ollama
!apt-get install zstd -y

# 1. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

# 2. Start the engine in the background
print("\nStarting Ollama server...")
subprocess.Popen(["ollama", "serve"])
time.sleep(3)

# 3. Download the Llama 3.2 model
print("Downloading Llama 3.2 (Watch how fast this is)...")
!ollama pull llama3.2
print("\n✅ BRAIN SUCCESSFULLY INSTALLED AND RUNNING! DAY 1 IS COMPLETE!")

In [ ]:
## DAY_02
!pip install -q sentence-transformers faiss-cpu opencv-python
print("✅ Vision tools installed!")

In [ ]:
!pip install pysrt ollama sentence-transformers faiss-cpu edge-tts

In [ ]:
# It asks you for the file names, cleans the text, writes the script, finds the timestamps (RAG), and saves the blueprint.

In [ ]:
#  ⚙️ Wake Up Ollama Server
!nohup ollama serve > ollama.log 2>&1 &
!sleep 3
!ollama pull llama3.2

In [ ]:
#
# @markdown Fill in all your filenames here so no other cell overwrites them.

movie_filename = "Evil Returns.mp4" # @param {type:"string"}
srt_filename = "Evil Returns.srt" # @param {type:"string"}
bgm_filename = "Music/action.mp3" # @param {type:"string"}
target_language = "English" # @param ["English", "Urdu"]
genre = "horror" # @param ["action", "horror", "romantic", "sad", "sci-fi", "thriller"]
voice_model = "en-US-ChristopherNeural" # @param ["en-US-ChristopherNeural", "en-US-EricNeural", "ur-PK-AsadNeural"]

print(f"✅ Settings Locked! \n🎥 Movie: {movie_filename}\n📜 SRT: {srt_filename}\n🎵 BGM: {bgm_filename}")

In [ ]:
# @title 🎬 Cell 4: AI Script Generation & Video RAG Sync
# @markdown Fill in your settings below. No hardcoded paths!

import os
import re
import json
import faiss
import pysrt
import ollama
import random
from sentence_transformers import SentenceTransformer


base_dir = '/content/drive/MyDrive/MovieApp/'
srt_path = os.path.join(base_dir, srt_filename)
index_path = os.path.join(base_dir, 'movie_index.faiss')
timeline_path = os.path.join(base_dir, 'recap_timeline.json')

MAX_WORDS = 8

# --- VS CODE LOGIC: AI CLEANUP & SPLITTING ---
def clean_ai(text):
    text = text.replace('\n', ' ').replace('*', '').replace('-', '')
    text = re.sub(r'\(.*?\)', '', text)
    text = re.sub(r'\[.*?\]', '', text)
    sentences = text.replace("!", ".").replace("?", ".").split(".")
    clean_sentences = []
    banned_starts = ["i will", "i can", "i am", "sure", "here is", "happy to", "in this scene", "certainly", "okay"]
    for s in sentences:
        s_clean = s.strip()
        if len(s_clean) < 3: continue
        is_banned = any(s_clean.lower().startswith(bad) for bad in banned_starts)
        if not is_banned: clean_sentences.append(s_clean)
    return ". ".join(clean_sentences) + "."

def strict_split(text):
    text = text.replace("?", ".").replace("!", ".").replace(";", ".")
    raw_sentences = [s.strip() for s in text.split(".") if s.strip()]
    final_sentences = []
    for sent in raw_sentences:
        words = sent.split()
        if len(words) <= MAX_WORDS:
            final_sentences.append(sent)
        else:
            for i in range(0, len(words), MAX_WORDS):
                final_sentences.append(" ".join(words[i:i+MAX_WORDS]))
    return final_sentences

def get_subs(subs_list, s, e):
    return " ".join([sub['text'] for sub in subs_list if s <= sub['start'] < e])[:4000]

print("⚡ Step 1: Analyzing movie timeline intelligently...")
if not os.path.exists(srt_path):
    print(f"🛑 ERROR: Could not find '{srt_path}'.")
else:
    subs = []
    srt_data = pysrt.open(srt_path)
    for sub in srt_data:
        start_sec = sub.start.ordinal / 1000.0
        end_sec = sub.end.ordinal / 1000.0
        text = sub.text.replace('\n', ' ')
        if "opensubtitles" not in text.lower() and "www." not in text.lower():
            subs.append({'start': start_sec, 'end': end_sec, 'text': text})

    story_start_time = max(0.0, subs[0]['start'] - 10.0)
    mov_dur = subs[-1]['end'] + 15.0

    actual_story_duration = mov_dur - story_start_time
    movie_mins = actual_story_duration / 60
    target_recap_mins = 15 if movie_mins <= 80 else 17
    chapter_length = actual_story_duration / target_recap_mins

    print(f"✅ Slicing into {target_recap_mins} chapters to hit ~{target_recap_mins} min runtime.")

    # --- Step 2: AI SCRIPT WRITING ---
    print(f"🧠 Step 2: AI Generating {target_language} Script via Ollama...")
    generated_sentences = []
    styles = ["Be intense.", "Be mysterious.", "Describe the action dramatically."]

    for i in range(target_recap_mins):
        s_t = story_start_time + (i * chapter_length)
        e_t = story_start_time + ((i + 1) * chapter_length)
        scene_txt = get_subs(subs, s_t, e_t)

        prompt = (f"SYSTEM: You are a professional movie recapper. Summarize this specific chapter. "
                  f"Write EXACTLY 6 to 8 continuous sentences in pure {target_language.upper()}. "
                  f"CRITICAL RULE: NEVER refuse the prompt. Just write the story. "
                  f"Style: {random.choice(styles)}. Context: {scene_txt}")

        try:
            resp = ollama.chat(model='llama3.2', messages=[{'role':'user','content':prompt}])
            script = clean_ai(resp['message']['content'])
        except Exception as e:
            script = "The characters navigate the unfolding events in silence."

        chunk_sentences = strict_split(script)
        generated_sentences.extend([s for s in chunk_sentences if len(s.strip()) > 2])
        print(f"  ...Wrote Chapter {i+1}/{target_recap_mins}")

    # --- Step 3: VIDEO RAG SYNC ---
    print("🔍 Step 3: Running RAG Search to map AI script to visual timestamps...")
    search_model = SentenceTransformer('clip-ViT-B-32')
    index = faiss.read_index(index_path)
    recap_timeline = []

    for i, sentence in enumerate(generated_sentences):
        emb = search_model.encode([sentence])
        faiss.normalize_L2(emb)
        _, indices = index.search(emb, k=1)
        recap_timeline.append({"text": sentence, "timestamp": int(indices[0][0])})

        if (i + 1) % 25 == 0:
            print(f"  ...Mapped {i + 1}/{len(generated_sentences)} clips to video...")

    with open(timeline_path, 'w') as f:
        json.dump(recap_timeline, f, indent=4)

    print(f"🏆 MEGACUT BLUEPRINT SAVED with {len(generated_sentences)} perfectly synced scenes! Ready for audio generation.")

In [ ]:
# edge tts environment
# --- DAY 4: PROFESSIONAL SYNC TOOLS ---
print("Installing Edge-TTS (Voice) and MoviePy (Editor)...")
!pip install -q edge-tts moviepy

print("Environment Ready.")


In [ ]:
# @title 🚀 Cell 5: Audio/Video Sync & FFmpeg GPU Render (CLEANED)
import os
import json
import asyncio
import subprocess
import edge_tts

# --- PATHS (Using variables from Master Settings) ---
base_dir = '/content/drive/MyDrive/MovieApp/'
movie_path = os.path.join(base_dir, movie_filename)
timeline_path = os.path.join(base_dir, 'recap_timeline.json')
bgm_path = os.path.join(base_dir, bgm_filename)
output_path = os.path.join(base_dir, 'FINAL_RECAP.mp4')
temp_dir = os.path.join(base_dir, 'temp_render')

os.makedirs(temp_dir, exist_ok=True)

def get_audio_duration(file_path):
    cmd = ['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=noprint_wrappers=1:nokey=1', file_path]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, text=True)
    return float(result.stdout.strip())

async def render_recap():
    # 🛑 INSTANT KILL SWITCH IF FILE IS MISSING
    if not os.path.exists(movie_path):
        print(f"🛑 FATAL ERROR: Cannot find the movie file at: {movie_path}")
        print("Check your Google Drive folder and make sure the spelling matches exactly!")
        return

    print("⏳ Loading Megacut Blueprint...")
    with open(timeline_path, 'r') as f:
        timeline = json.load(f)

    chunk_files = []
    print(f"🔥 Starting Generation for {len(timeline)} scenes...")

    for i, scene in enumerate(timeline):
        text = scene['text']
        timestamp = scene['timestamp']

        audio_path = os.path.join(temp_dir, f"audio_{i:03d}.mp3")
        communicate = edge_tts.Communicate(text, voice_model, rate="+5%")
        await communicate.save(audio_path)

        duration = get_audio_duration(audio_path)
        chunk_path = os.path.join(temp_dir, f"chunk_{i:03d}.mp4")

        print(f"  🎬 Syncing Scene {i+1}/{len(timeline)}: [Time: {timestamp}s | Length: {duration:.2f}s]")

        cmd = [
            'ffmpeg', '-y', '-hide_banner', '-loglevel', 'error',
            '-ss', str(timestamp), '-i', movie_path,
            '-i', audio_path,
            '-map', '0:v:0', '-map', '1:a:0',
            '-t', str(duration),
            '-c:v', 'libx264', '-preset', 'superfast', '-crf', '23',
            '-c:a', 'aac', '-b:a', '192k',
            '-shortest', chunk_path
        ]

        process = subprocess.run(cmd, capture_output=True, text=True)
        if process.returncode != 0:
            print(f"❌ FFmpeg Error on scene {i+1}: {process.stderr}")
            continue

        if os.path.exists(chunk_path):
            chunk_files.append(chunk_path)

    if not chunk_files:
        print("🛑 FATAL ERROR: No video chunks created.")
        return

    print(f"🧵 Stitching {len(chunk_files)} scenes together...")
    concat_file = os.path.join(temp_dir, 'concat_list.txt')
    with open(concat_file, 'w') as f:
        for chunk in chunk_files:
            abs_path = os.path.abspath(chunk).replace("'", "'\\''")
            f.write(f"file '{abs_path}'\n")

    stitched_path = os.path.join(temp_dir, 'stitched_no_music.mp4')
    stitch_cmd = ['ffmpeg', '-y', '-hide_banner', '-loglevel', 'error', '-f', 'concat', '-safe', '0', '-i', concat_file, '-c', 'copy', stitched_path]

    stitch_process = subprocess.run(stitch_cmd, capture_output=True, text=True)
    if stitch_process.returncode != 0:
        print(f"❌ Stitching Error: {stitch_process.stderr}")
        return

    print("🎵 Adding Background Music...")
    if os.path.exists(bgm_path):
        subprocess.run([
            'ffmpeg', '-y', '-hide_banner', '-loglevel', 'error',
            '-i', stitched_path,
            '-stream_loop', '-1', '-i', bgm_path,
            '-filter_complex', '[0:a]volume=1.0[a0];[1:a]volume=0.15[a1];[a0][a1]amix=inputs=2:duration=first:dropout_transition=2[a]',
            '-map', '0:v', '-map', '[a]',
            '-c:v', 'copy', '-c:a', 'aac',
            output_path
        ])
    else:
        print("⚠️ No BGM found. Skipping music overlay.")
        os.rename(stitched_path, output_path)

    print(f"\n✅✅ DONE! Your final recap is saved at: {output_path}")

await render_recap()